#### Strategy Outline

Bias for commercial net long > 80 and <20 for net short
When bias is long and RSI is <30 go long, go short when bias is short and rsi is  over 70 go short

Exit: RSI @60 or 20 day limit

Risk management: 2 ATR stop, 3 ATR target, 1% risk per trade

In [22]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')

with open('cot_data.json', 'r') as f:
    data__raw = json.load(f)

df = pd.DataFrame(data__raw)
df["Date"] = pd.to_datetime(df["Date"], unit='ms')  # Convert from milliseconds


#### Prepare data for analysis

In [23]:
def prepare_strategy_data(df, market_name):
    """
    Prepare strategy data for a specific market.
    Combines daily OHLC price data with weekly COT data.
    
    Returns DataFrame with: Date, Open, High, Low, Close, RSI, Commercial_Index, OI
    """
    market_data = df[df['Market'] == market_name].copy()

    cot_weekly = market_data[market_data['data_type'] == 'weekly_cot'].copy()
    price_daily = market_data[market_data['data_type'] == 'daily_price'].copy()

    if price_daily.empty:
        print(f"⚠ No daily price data for {market_name}")
        return pd.DataFrame()  # Return empty DataFrame instead of COT-only data

    # Include OHLC columns for ATR calculation
    price_cols = ['Date', 'Open', 'High', 'Low', 'Close', 'RSI']
    
    # Check which columns actually exist
    available_cols = [col for col in price_cols if col in price_daily.columns]
    missing_cols = [col for col in price_cols if col not in price_daily.columns]
    
    if missing_cols:
        print(f"⚠ {market_name}: Missing columns: {missing_cols}")
    
    strategy_data = price_daily[available_cols].copy()

    # Merge weekly COT data directly, then forward fill
    cot_cols = ['Net Commercial Position', 'OI', 'Commercial_Index']
    cot_for_merge = cot_weekly[['Date'] + cot_cols].copy()
    
    strategy_data = pd.merge(strategy_data, cot_for_merge, on='Date', how='left')
    
    # Forward fill COT values to fill gaps between weekly reports
    strategy_data[cot_cols] = strategy_data[cot_cols].ffill()
    
    # Remove rows with missing critical data (need Close and Commercial_Index at minimum)
    strategy_data = strategy_data.dropna(subset=['Close', 'Commercial_Index'])
    
    # Sort by date and reset index
    strategy_data = strategy_data.sort_values('Date').reset_index(drop=True)
    
    print(f"✓ {market_name}: {len(strategy_data)} rows, columns: {list(strategy_data.columns)}")
    
    return strategy_data 



In [24]:
# Test the function with a market
test_market = "GOLD - COMMODITY EXCHANGE INC."
test_data = prepare_strategy_data(df, test_market)

# Display first few rows to verify OHLC is included
print(f"\nFirst 5 rows:")
print(test_data.head())
print(f"\nLast 5 rows:")
print(test_data.tail())

✓ GOLD - COMMODITY EXCHANGE INC.: 502 rows, columns: ['Date', 'Open', 'High', 'Low', 'Close', 'RSI', 'Net Commercial Position', 'OI', 'Commercial_Index']

First 5 rows:
        Date         Open         High          Low        Close  RSI  \
0 2024-01-02  2063.500000  2073.699951  2057.100098  2064.399902  NaN   
1 2024-01-03  2034.199951  2044.000000  2034.199951  2034.199951  NaN   
2 2024-01-04  2041.599976  2044.500000  2038.000000  2042.300049  NaN   
3 2024-01-05  2044.500000  2048.100098  2042.400024  2042.400024  NaN   
4 2024-01-08  2019.099976  2033.699951  2019.099976  2026.599976  NaN   

   Net Commercial Position        OI  Commercial_Index  
0                -235478.0  500364.0         55.818686  
1                -235478.0  500364.0         55.818686  
2                -235478.0  500364.0         55.818686  
3                -235478.0  500364.0         55.818686  
4                -235478.0  500364.0         55.818686  

Last 5 rows:
          Date         Open         

### Technical Indicators

In [25]:
def calculate_atr(data, period=10):

    df = data.copy()
    
    # Validate required columns exist
    required = ['High', 'Low', 'Close']
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns for ATR: {missing}")
    
    # Calculate True Range components
    df['prev_close'] = df['Close'].shift(1)
    
    df['tr1'] = df['High'] - df['Low']                    # Current bar range
    df['tr2'] = abs(df['High'] - df['prev_close'])        # High vs previous close
    df['tr3'] = abs(df['Low'] - df['prev_close'])         # Low vs previous close
    
    # True Range is the maximum of the three
    df['TR'] = df[['tr1', 'tr2', 'tr3']].max(axis=1)
    
    # ATR is the rolling mean of True Range
    df['ATR'] = df['TR'].rolling(window=period).mean()
    
    # Clean up temporary columns
    df.drop(columns=['prev_close', 'tr1', 'tr2', 'tr3', 'TR'], inplace=True)
    
    # Report ATR coverage
    valid_atr = df['ATR'].notna().sum()
    total_rows = len(df)
    print(f"✓ ATR calculated: {valid_atr}/{total_rows} rows have valid ATR (first {period} rows are NaN)")
    
    return df


# Test ATR calculation
test_data_with_atr = calculate_atr(test_data.copy())
print(f"\nATR sample (last 5 rows):")
print(test_data_with_atr[['Date', 'Close', 'High', 'Low', 'ATR']].tail())


✓ ATR calculated: 493/502 rows have valid ATR (first 10 rows are NaN)

ATR sample (last 5 rows):
          Date        Close         High          Low        ATR
497 2025-12-22  4444.600098  4447.600098  4371.100098  56.050000
498 2025-12-23  4482.799805  4503.799805  4450.399902  57.769971
499 2025-12-24  4480.600098  4503.399902  4468.399902  56.179980
500 2025-12-26  4529.100098  4556.299805  4502.000000  54.699951
501 2025-12-29  4325.100098  4379.000000  4325.100098  65.599951


In [26]:
def generate_signals(data, commercial_long_threshold=80, commercial_short_threshold=20, 
                     rsi_oversold=30, rsi_overbought=70):
    """
    Generate trading signals based on COT Commercial Index and RSI.
    
    Strategy Rules:
    - LONG:  Commercial_Index >= 80 AND RSI < 30
    - SHORT: Commercial_Index <= 20 AND RSI > 70
    
    Args:
        data: DataFrame with 'Commercial_Index' and 'RSI' columns
        commercial_long_threshold: Commercial Index level for long bias (default 80)
        commercial_short_threshold: Commercial Index level for short bias (default 20)
        rsi_oversold: RSI level for oversold (default 30)
        rsi_overbought: RSI level for overbought (default 70)
    
    Returns:
        DataFrame with 'signal' column added (1=Long, -1=Short, 0=None)
    """
    df = data.copy()
    
    # Validate required columns
    required = ['Commercial_Index', 'RSI']
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns for signals: {missing}")
    
    # Initialize signal column
    df['signal'] = 0
    
    # Long signal: Commercial bullish (>=80) AND RSI oversold (<30)
    long_condition = (df['Commercial_Index'] >= commercial_long_threshold) & (df['RSI'] < rsi_oversold)
    df.loc[long_condition, 'signal'] = 1
    
    # Short signal: Commercial bearish (<=20) AND RSI overbought (>70)
    short_condition = (df['Commercial_Index'] <= commercial_short_threshold) & (df['RSI'] > rsi_overbought)
    df.loc[short_condition, 'signal'] = -1
    
    # Count signals
    long_signals = (df['signal'] == 1).sum()
    short_signals = (df['signal'] == -1).sum()
    total_rows = len(df)
    
    print(f"✓ Signals generated:")
    print(f"   Long signals:  {long_signals} (Commercial >= {commercial_long_threshold} AND RSI < {rsi_oversold})")
    print(f"   Short signals: {short_signals} (Commercial <= {commercial_short_threshold} AND RSI > {rsi_overbought})")
    print(f"   No signal:     {total_rows - long_signals - short_signals}")
    
    return df


# Test signal generation
test_data_with_signals = generate_signals(test_data_with_atr.copy())

# Show rows where we have signals
signal_rows = test_data_with_signals[test_data_with_signals['signal'] != 0]
if not signal_rows.empty:
    print(f"\nSignal dates:")
    print(signal_rows[['Date', 'Close', 'RSI', 'Commercial_Index', 'signal']].to_string())
else:
    print("\n⚠ No signals found for this market with default thresholds")


✓ Signals generated:
   Long signals:  2 (Commercial >= 80 AND RSI < 30)
   Short signals: 50 (Commercial <= 20 AND RSI > 70)
   No signal:     450

Signal dates:
          Date        Close         RSI  Commercial_Index  signal
29  2024-02-13  1992.900024   20.342687        100.000000       1
30  2024-02-14  1990.300049   19.269853        100.000000       1
134 2024-07-16  2462.399902   99.002992         16.281610      -1
135 2024-07-17  2454.800049   92.029786         16.281610      -1
136 2024-07-18  2451.800049   89.540249         16.281610      -1
159 2024-08-20  2511.300049   72.781686         12.424622      -1
160 2024-08-21  2508.399902   70.736369         12.424622      -1
175 2024-09-12  2551.199951   79.708618         17.569594      -1
176 2024-09-13  2581.300049   84.828262         17.569594      -1
177 2024-09-16  2580.399902   99.749748         17.569594      -1
178 2024-09-17  2564.300049   83.020994          2.452256      -1
179 2024-09-18  2570.699951   84.082169      

In [ ]:
class COTRSIBacktester:
    """
    Backtester for COT + RSI trading strategy.
    
    Key Features:
    - Ignores duplicate signals while position is open (no stacking)
    - Exits on RSI cross (60), 20-day limit, stop loss, or take profit
    - Position sizing based on 1% risk per trade
    - Tracks all trades with entry/exit details
    """
    
    def __init__(self, initial_capital=30000, risk_per_trade=0.01, 
                 atr_stop_mult=2, atr_target_mult=3, max_hold_days=20, rsi_exit=60):
        """
        Initialize backtester.
        
        Args:
            initial_capital: Starting capital (default $30,000)
            risk_per_trade: Fraction of capital to risk per trade (default 1%)
            atr_stop_mult: ATR multiplier for stop loss (default 2x)
            atr_target_mult: ATR multiplier for take profit (default 3x)
            max_hold_days: Maximum days to hold a trade (default 20)
            rsi_exit: RSI level to exit trades (default 60)
        """
        self.initial_capital = initial_capital
        self.risk_per_trade = risk_per_trade
        self.atr_stop_mult = atr_stop_mult
        self.atr_target_mult = atr_target_mult
        self.max_hold_days = max_hold_days
        self.rsi_exit = rsi_exit
        
        # State tracking
        self.capital = initial_capital
        self.trades = []
        self.equity_curve = []
        
    def calculate_position_size(self, entry_price, atr, market_name=""):
        """
        Position sizing based on 1% risk rule.
        
        Formula:
        - Risk amount = capital × 0.01
        - Stop distance = 2 × ATR
        - Max units = risk_amount / stop_distance
        - If units < 1: missed trade (can't afford 1 unit)
        - If units >= 1: round to nearest integer
        
        Exception: Bitcoin and Ether allow fractional units
        """
        if pd.isna(atr) or atr <= 0:
            return 0, f"Invalid ATR: {atr}"
        
        risk_amount = self.capital * self.risk_per_trade  # 1% of capital
        stop_distance = self.atr_stop_mult * atr          # 2 × ATR
        
        raw_units = risk_amount / stop_distance
        
        # Bitcoin and Ether can be traded as fractional units
        allows_fractional = 'BITCOIN' in market_name.upper() or 'ETHER' in market_name.upper()
        
        if allows_fractional:
            # Allow fractional units for crypto (round to 4 decimal places)
            if raw_units < 0.0001:
                return 0, f"Missed trade: Position too small ({raw_units:.6f} units)"
            return round(raw_units, 4), None
        else:
            # Standard assets require whole units
            if raw_units < 1:
                return 0, f"Missed trade: Can only afford {raw_units:.2f} units"
            return round(raw_units), None
    
    def backtest(self, data, market_name="Unknown"):
        """
        Run backtest on prepared data.
        
        Args:
            data: DataFrame with Date, Open, High, Low, Close, RSI, ATR, signal columns
            market_name: Name of market for reporting
            
        Returns:
            dict with trades list and equity curve
        """
        df = data.copy().reset_index(drop=True)
        
        # Validate required columns
        required = ['Date', 'Open', 'Close', 'RSI', 'ATR', 'signal']
        missing = [col for col in required if col not in df.columns]
        if missing:
            raise ValueError(f"Missing columns: {missing}")
        
        # State variables
        in_position = False
        position_direction = 0  # 1=long, -1=short
        entry_price = 0
        entry_date = None
        entry_idx = 0
        stop_loss = 0
        take_profit = 0
        units = 0
        
        trades = []
        equity = [self.initial_capital]
        current_capital = self.initial_capital
        
        for i in range(len(df)):
            row = df.iloc[i]
            date = row['Date']
            open_price = row['Open']
            high = row['High'] if 'High' in df.columns else row['Close']
            low = row['Low'] if 'Low' in df.columns else row['Close']
            close = row['Close']
            rsi = row['RSI']
            atr = row['ATR']
            signal = row['signal']
            
            # Skip if missing critical data
            if pd.isna(close) or pd.isna(rsi):
                equity.append(current_capital)
                continue
            
            # ===== CHECK EXITS FIRST =====
            if in_position:
                days_held = i - entry_idx
                exit_reason = None
                exit_price = None
                
                if position_direction == 1:  # Long position
                    # Check stop loss (hit during day)
                    if low <= stop_loss:
                        exit_reason = "Stop Loss"
                        exit_price = stop_loss
                    # Check take profit (hit during day)
                    elif high >= take_profit:
                        exit_reason = "Take Profit"
                        exit_price = take_profit
                    # Check RSI exit (close >= 60)
                    elif rsi >= self.rsi_exit:
                        exit_reason = f"RSI Exit ({rsi:.1f})"
                        exit_price = close
                    # Check max hold days
                    elif days_held >= self.max_hold_days:
                        exit_reason = f"Max Hold ({days_held} days)"
                        exit_price = close
                        
                else:  # Short position
                    # Check stop loss (hit during day)
                    if high >= stop_loss:
                        exit_reason = "Stop Loss"
                        exit_price = stop_loss
                    # Check take profit (hit during day)
                    elif low <= take_profit:
                        exit_reason = "Take Profit"
                        exit_price = take_profit
                    # Check RSI exit (close <= 60 for shorts... wait, strategy says RSI @60)
                    # For shorts, we exit when RSI drops back to 60 (from overbought)
                    elif rsi <= self.rsi_exit:
                        exit_reason = f"RSI Exit ({rsi:.1f})"
                        exit_price = close
                    # Check max hold days
                    elif days_held >= self.max_hold_days:
                        exit_reason = f"Max Hold ({days_held} days)"
                        exit_price = close
                
                # Execute exit if triggered
                if exit_reason:
                    if position_direction == 1:
                        pnl = (exit_price - entry_price) * units
                    else:
                        pnl = (entry_price - exit_price) * units
                    
                    pnl_pct = (pnl / (entry_price * units)) * 100 if units > 0 else 0
                    current_capital += pnl
                    
                    trades.append({
                        'market': market_name,
                        'entry_date': entry_date,
                        'exit_date': date,
                        'direction': 'Long' if position_direction == 1 else 'Short',
                        'entry_price': entry_price,
                        'exit_price': exit_price,
                        'units': units,
                        'pnl': pnl,
                        'pnl_pct': pnl_pct,
                        'exit_reason': exit_reason,
                        'days_held': days_held
                    })
                    
                    in_position = False
                    position_direction = 0
            
            # ===== CHECK ENTRIES (only if not in position) =====
            if not in_position and signal != 0 and not pd.isna(atr):
                # Enter at next day's open (use current close as proxy)
                entry_price = close
                entry_date = date
                entry_idx = i
                position_direction = signal
                
                # Calculate position size
                units, error = self.calculate_position_size(entry_price, atr, market_name)
                
                if error:
                    # Skip this signal - can't size position
                    continue
                
                # Set stop loss and take profit
                if signal == 1:  # Long
                    stop_loss = entry_price - (self.atr_stop_mult * atr)
                    take_profit = entry_price + (self.atr_target_mult * atr)
                else:  # Short
                    stop_loss = entry_price + (self.atr_stop_mult * atr)
                    take_profit = entry_price - (self.atr_target_mult * atr)
                
                in_position = True
            
            equity.append(current_capital)
        
        # Close any open position at end
        if in_position:
            final_close = df.iloc[-1]['Close']
            final_date = df.iloc[-1]['Date']
            days_held = len(df) - 1 - entry_idx
            
            if position_direction == 1:
                pnl = (final_close - entry_price) * units
            else:
                pnl = (entry_price - final_close) * units
            
            pnl_pct = (pnl / (entry_price * units)) * 100 if units > 0 else 0
            current_capital += pnl
            
            trades.append({
                'market': market_name,
                'entry_date': entry_date,
                'exit_date': final_date,
                'direction': 'Long' if position_direction == 1 else 'Short',
                'entry_price': entry_price,
                'exit_price': final_close,
                'units': units,
                'pnl': pnl,
                'pnl_pct': pnl_pct,
                'exit_reason': 'End of Data',
                'days_held': days_held
            })
            equity.append(current_capital)
        
        self.trades = trades
        self.equity_curve = equity
        self.capital = current_capital
        
        return {
            'trades': pd.DataFrame(trades),
            'equity_curve': equity,
            'final_capital': current_capital,
            'total_return': (current_capital - self.initial_capital) / self.initial_capital * 100
        }


# Test the backtester
print("=" * 80)
print("BACKTESTER TEST")
print("=" * 80)

backtester = COTRSIBacktester(
    initial_capital=30000,
    risk_per_trade=0.01,
    atr_stop_mult=2,
    atr_target_mult=3,
    max_hold_days=20,
    rsi_exit=60
)

results = backtester.backtest(test_data_with_signals, market_name=test_market)

print(f"\n📊 Results for {test_market}:")
print(f"   Initial Capital: ${backtester.initial_capital:,.2f}")
print(f"   Final Capital:   ${results['final_capital']:,.2f}")
print(f"   Total Return:    {results['total_return']:.2f}%")
print(f"   Total Trades:    {len(results['trades'])}")

if not results['trades'].empty:
    print(f"\n📝 Trade Summary:")
    print(results['trades'][['entry_date', 'exit_date', 'direction', 'entry_price', 
                             'exit_price', 'pnl', 'exit_reason']].to_string())


BACKTESTER TEST

📊 Results for GOLD - COMMODITY EXCHANGE INC.:
   Initial Capital: $30,000.00
   Final Capital:   $29,096.26
   Total Return:    -3.01%
   Total Trades:    15

📝 Trade Summary:
   entry_date  exit_date direction  entry_price   exit_price         pnl      exit_reason
0  2024-02-13 2024-02-23      Long  1992.900024  2038.599976  319.899658  RSI Exit (79.4)
1  2024-07-16 2024-07-19     Short  2462.399902  2395.500000  334.499512  RSI Exit (58.2)
2  2024-08-20 2024-08-22     Short  2511.300049  2478.899902  162.000732  RSI Exit (55.0)
3  2024-09-12 2024-09-20     Short  2551.199951  2598.879883 -286.079590        Stop Loss
4  2024-09-20 2024-09-25     Short  2619.899902  2663.619922 -262.320117        Stop Loss
5  2024-09-25 2024-10-04     Short  2659.199951  2645.800049   80.399414  RSI Exit (51.2)
6  2024-10-17 2024-10-21     Short  2691.000000  2733.179980 -295.259863        Stop Loss
7  2024-10-21 2024-10-29     Short  2723.100098  2766.660107 -261.360059        Stop Lo

In [28]:
def calculate_performance_metrics(trades_df, equity_curve, initial_capital=30000):
    """
    Calculate comprehensive performance metrics for a backtest.
    
    Args:
        trades_df: DataFrame with trade records (must have 'pnl' column)
        equity_curve: List of equity values over time
        initial_capital: Starting capital
    
    Returns:
        dict with performance metrics
    """
    metrics = {}
    
    if trades_df.empty:
        return {
            'total_trades': 0,
            'win_rate': 0,
            'profit_factor': 0,
            'total_return_pct': 0,
            'cagr': 0,
            'max_drawdown_pct': 0,
            'sharpe_ratio': 0,
            'avg_win': 0,
            'avg_loss': 0,
            'largest_win': 0,
            'largest_loss': 0,
            'avg_days_held': 0
        }
    
    # Basic trade stats
    total_trades = len(trades_df)
    winning_trades = trades_df[trades_df['pnl'] > 0]
    losing_trades = trades_df[trades_df['pnl'] < 0]
    
    win_count = len(winning_trades)
    loss_count = len(losing_trades)
    
    metrics['total_trades'] = total_trades
    metrics['winning_trades'] = win_count
    metrics['losing_trades'] = loss_count
    metrics['win_rate'] = (win_count / total_trades * 100) if total_trades > 0 else 0
    
    # P&L stats
    total_profit = winning_trades['pnl'].sum() if not winning_trades.empty else 0
    total_loss = abs(losing_trades['pnl'].sum()) if not losing_trades.empty else 0
    
    metrics['gross_profit'] = total_profit
    metrics['gross_loss'] = total_loss
    metrics['net_profit'] = total_profit - total_loss
    metrics['profit_factor'] = (total_profit / total_loss) if total_loss > 0 else float('inf')
    
    # Average trade stats
    metrics['avg_win'] = winning_trades['pnl'].mean() if not winning_trades.empty else 0
    metrics['avg_loss'] = losing_trades['pnl'].mean() if not losing_trades.empty else 0
    metrics['avg_trade'] = trades_df['pnl'].mean()
    
    # Largest trades
    metrics['largest_win'] = winning_trades['pnl'].max() if not winning_trades.empty else 0
    metrics['largest_loss'] = losing_trades['pnl'].min() if not losing_trades.empty else 0
    
    # Days held
    if 'days_held' in trades_df.columns:
        metrics['avg_days_held'] = trades_df['days_held'].mean()
    
    # Return metrics
    final_capital = equity_curve[-1] if equity_curve else initial_capital
    total_return = (final_capital - initial_capital) / initial_capital
    metrics['total_return_pct'] = total_return * 100
    
    # CAGR (Compound Annual Growth Rate)
    # Assume ~252 trading days per year
    if 'entry_date' in trades_df.columns and 'exit_date' in trades_df.columns:
        first_date = trades_df['entry_date'].min()
        last_date = trades_df['exit_date'].max()
        days_traded = (last_date - first_date).days
        years = days_traded / 365.25 if days_traded > 0 else 1
    else:
        years = len(equity_curve) / 252  # Approximate
    
    if years > 0 and final_capital > 0:
        metrics['cagr'] = ((final_capital / initial_capital) ** (1 / years) - 1) * 100
    else:
        metrics['cagr'] = 0
    
    # Max Drawdown
    equity_series = pd.Series(equity_curve)
    rolling_max = equity_series.cummax()
    drawdown = (equity_series - rolling_max) / rolling_max
    metrics['max_drawdown_pct'] = abs(drawdown.min()) * 100
    
    # Sharpe Ratio (simplified - using trade returns)
    if len(trades_df) > 1:
        trade_returns = trades_df['pnl_pct'] / 100 if 'pnl_pct' in trades_df.columns else trades_df['pnl'] / initial_capital
        avg_return = trade_returns.mean()
        std_return = trade_returns.std()
        # Annualize assuming ~20 trades per year (rough estimate)
        trades_per_year = 252 / trades_df['days_held'].mean() if 'days_held' in trades_df.columns else 20
        metrics['sharpe_ratio'] = (avg_return * trades_per_year) / (std_return * np.sqrt(trades_per_year)) if std_return > 0 else 0
    else:
        metrics['sharpe_ratio'] = 0
    
    return metrics


# Test performance metrics
metrics = calculate_performance_metrics(results['trades'], results['equity_curve'], backtester.initial_capital)

print("=" * 80)
print("PERFORMANCE METRICS")
print("=" * 80)
print(f"\n📊 Trade Statistics:")
print(f"   Total Trades:    {metrics['total_trades']}")
print(f"   Winning Trades:  {metrics['winning_trades']}")
print(f"   Losing Trades:   {metrics['losing_trades']}")
print(f"   Win Rate:        {metrics['win_rate']:.1f}%")

print(f"\n💰 Profit/Loss:")
print(f"   Gross Profit:    ${metrics['gross_profit']:,.2f}")
print(f"   Gross Loss:      ${metrics['gross_loss']:,.2f}")
print(f"   Net Profit:      ${metrics['net_profit']:,.2f}")
print(f"   Profit Factor:   {metrics['profit_factor']:.2f}")

print(f"\n📈 Returns:")
print(f"   Total Return:    {metrics['total_return_pct']:.2f}%")
print(f"   CAGR:            {metrics['cagr']:.2f}%")
print(f"   Max Drawdown:    {metrics['max_drawdown_pct']:.2f}%")
print(f"   Sharpe Ratio:    {metrics['sharpe_ratio']:.2f}")

print(f"\n📝 Trade Details:")
print(f"   Avg Win:         ${metrics['avg_win']:,.2f}")
print(f"   Avg Loss:        ${metrics['avg_loss']:,.2f}")
print(f"   Largest Win:     ${metrics['largest_win']:,.2f}")
print(f"   Largest Loss:    ${metrics['largest_loss']:,.2f}")
print(f"   Avg Days Held:   {metrics['avg_days_held']:.1f}")


PERFORMANCE METRICS

📊 Trade Statistics:
   Total Trades:    15
   Winning Trades:  7
   Losing Trades:   8
   Win Rate:        46.7%

💰 Profit/Loss:
   Gross Profit:    $1,329.80
   Gross Loss:      $2,233.54
   Net Profit:      $-903.74
   Profit Factor:   0.60

📈 Returns:
   Total Return:    -3.01%
   CAGR:            -1.90%
   Max Drawdown:    6.30%
   Sharpe Ratio:    -1.49

📝 Trade Details:
   Avg Win:         $189.97
   Avg Loss:        $-279.19
   Largest Win:     $334.50
   Largest Loss:    $-297.52
   Avg Days Held:   4.1


In [29]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def create_strategy_chart(data, trades_df, market_name):
    """
    Create a 3-pane chart showing Price, Commercial Index, and RSI with trade markers.
    
    Args:
        data: DataFrame with Date, Close, Commercial_Index, RSI, signal columns
        trades_df: DataFrame with trade records (entry_date, exit_date, direction, etc.)
        market_name: Name of market for title
    
    Returns:
        Plotly figure object
    """
    df = data.copy()
    
    # Create figure with 3 subplots
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.5, 0.25, 0.25],
        subplot_titles=(
            f"Price: {market_name}",
            "Commercial Index (COT)",
            "RSI (14)"
        )
    )
    
    # ===== PANE 1: PRICE =====
    fig.add_trace(
        go.Scatter(
            x=df['Date'],
            y=df['Close'],
            name="Close Price",
            line=dict(color="#2962FF", width=1.5)
        ),
        row=1, col=1
    )
    
    # Add entry markers from trades
    if not trades_df.empty:
        # Long entries (green triangle up)
        long_trades = trades_df[trades_df['direction'] == 'Long']
        if not long_trades.empty:
            fig.add_trace(
                go.Scatter(
                    x=long_trades['entry_date'],
                    y=long_trades['entry_price'],
                    mode='markers',
                    name='Long Entry',
                    marker=dict(symbol='triangle-up', size=12, color='#00C853', line=dict(width=1, color='white'))
                ),
                row=1, col=1
            )
        
        # Short entries (red triangle down)
        short_trades = trades_df[trades_df['direction'] == 'Short']
        if not short_trades.empty:
            fig.add_trace(
                go.Scatter(
                    x=short_trades['entry_date'],
                    y=short_trades['entry_price'],
                    mode='markers',
                    name='Short Entry',
                    marker=dict(symbol='triangle-down', size=12, color='#FF1744', line=dict(width=1, color='white'))
                ),
                row=1, col=1
            )
        
        # Exit markers (X marks)
        fig.add_trace(
            go.Scatter(
                x=trades_df['exit_date'],
                y=trades_df['exit_price'],
                mode='markers',
                name='Exit',
                marker=dict(symbol='x', size=10, color='#FFD600', line=dict(width=2))
            ),
            row=1, col=1
        )
    
    # ===== PANE 2: COMMERCIAL INDEX =====
    fig.add_trace(
        go.Scatter(
            x=df['Date'],
            y=df['Commercial_Index'],
            name="Commercial Index",
            line=dict(color="#00BFA5", width=2)
        ),
        row=2, col=1
    )
    
    # Add threshold lines at 20 and 80
    fig.add_hline(y=80, line_dash="dash", line_color="orange", line_width=1, row=2, col=1)
    fig.add_hline(y=20, line_dash="dash", line_color="orange", line_width=1, row=2, col=1)
    
    # Add shaded zones
    fig.add_hrect(y0=80, y1=100, fillcolor="rgba(0,200,83,0.1)", line_width=0, row=2, col=1)
    fig.add_hrect(y0=0, y1=20, fillcolor="rgba(255,23,68,0.1)", line_width=0, row=2, col=1)
    
    # ===== PANE 3: RSI =====
    fig.add_trace(
        go.Scatter(
            x=df['Date'],
            y=df['RSI'],
            name="RSI",
            line=dict(color="#AA00FF", width=2)
        ),
        row=3, col=1
    )
    
    # RSI threshold lines
    fig.add_hline(y=70, line_dash="dash", line_color="red", line_width=1, row=3, col=1)
    fig.add_hline(y=60, line_dash="dot", line_color="gray", line_width=1, row=3, col=1)  # Exit level
    fig.add_hline(y=30, line_dash="dash", line_color="green", line_width=1, row=3, col=1)
    
    # RSI zones
    fig.add_hrect(y0=70, y1=100, fillcolor="rgba(255,23,68,0.1)", line_width=0, row=3, col=1)
    fig.add_hrect(y0=0, y1=30, fillcolor="rgba(0,200,83,0.1)", line_width=0, row=3, col=1)
    
    # ===== LAYOUT =====
    fig.update_layout(
        height=900,
        title=dict(
            text=f"COT + RSI Strategy Analysis: {market_name}",
            font=dict(size=18)
        ),
        hovermode="x unified",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        ),
        template="plotly_white"
    )
    
    # Y-axis labels
    fig.update_yaxes(title_text="Price ($)", row=1, col=1)
    fig.update_yaxes(title_text="Index", range=[0, 100], row=2, col=1)
    fig.update_yaxes(title_text="RSI", range=[0, 100], row=3, col=1)
    
    # X-axis with range selector
    fig.update_xaxes(
        title_text="Date",
        rangeslider_visible=False,
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1M", step="month", stepmode="backward"),
                dict(count=3, label="3M", step="month", stepmode="backward"),
                dict(count=6, label="6M", step="month", stepmode="backward"),
                dict(count=1, label="1Y", step="year", stepmode="backward"),
                dict(step="all", label="All")
            ]),
            bgcolor="white",
            font=dict(size=10)
        ),
        row=3, col=1
    )
    
    return fig


# Create and display the chart
strategy_chart = create_strategy_chart(test_data_with_signals, results['trades'], test_market)
strategy_chart.show()


In [30]:
def create_trade_table(trades_df, last_n=10):
    """
    Create a formatted Plotly table showing trade history.
    
    Args:
        trades_df: DataFrame with trade records
        last_n: Number of recent trades to show (default 10)
    
    Returns:
        Plotly figure object
    """
    if trades_df.empty:
        print("No trades to display")
        return None
    
    # Get last N trades
    display_trades = trades_df.tail(last_n).copy()
    
    # Format columns for display
    display_trades['entry_date'] = pd.to_datetime(display_trades['entry_date']).dt.strftime('%Y-%m-%d')
    display_trades['exit_date'] = pd.to_datetime(display_trades['exit_date']).dt.strftime('%Y-%m-%d')
    display_trades['entry_price'] = display_trades['entry_price'].apply(lambda x: f"${x:,.2f}")
    display_trades['exit_price'] = display_trades['exit_price'].apply(lambda x: f"${x:,.2f}")
    display_trades['pnl_formatted'] = display_trades['pnl'].apply(lambda x: f"${x:,.2f}")
    display_trades['pnl_pct_formatted'] = display_trades['pnl_pct'].apply(lambda x: f"{x:.2f}%")
    
    # Color code P&L
    pnl_colors = ['#00C853' if x > 0 else '#FF1744' if x < 0 else 'white' for x in display_trades['pnl']]
    
    # Create table
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=['Entry Date', 'Exit Date', 'Direction', 'Entry Price', 'Exit Price', 'P&L', 'P&L %', 'Exit Reason'],
            fill_color='#1a1a2e',
            font=dict(color='white', size=12),
            align='center',
            height=35
        ),
        cells=dict(
            values=[
                display_trades['entry_date'],
                display_trades['exit_date'],
                display_trades['direction'],
                display_trades['entry_price'],
                display_trades['exit_price'],
                display_trades['pnl_formatted'],
                display_trades['pnl_pct_formatted'],
                display_trades['exit_reason']
            ],
            fill_color=[
                ['#16213e'] * len(display_trades),  # Entry date
                ['#16213e'] * len(display_trades),  # Exit date
                [('#0f3460' if d == 'Long' else '#3d1a1a') for d in display_trades['direction']],  # Direction
                ['#16213e'] * len(display_trades),  # Entry price
                ['#16213e'] * len(display_trades),  # Exit price
                [('#1b4332' if p > 0 else '#4a1c1c' if p < 0 else '#16213e') for p in display_trades['pnl']],  # P&L
                [('#1b4332' if p > 0 else '#4a1c1c' if p < 0 else '#16213e') for p in display_trades['pnl']],  # P&L %
                ['#16213e'] * len(display_trades),  # Exit reason
            ],
            font=dict(color='white', size=11),
            align='center',
            height=30
        )
    )])
    
    fig.update_layout(
        title=dict(
            text=f"Last {len(display_trades)} Trades",
            font=dict(size=16, color='#333')
        ),
        height=400,
        margin=dict(l=20, r=20, t=50, b=20)
    )
    
    return fig


# Display trade table
trade_table = create_trade_table(results['trades'], last_n=10)
if trade_table:
    trade_table.show()


In [31]:
def create_metrics_summary(metrics, market_name):
    """
    Create a visual summary table of performance metrics.
    
    Args:
        metrics: dict with performance metrics
        market_name: Name of market for title
    
    Returns:
        Plotly figure object
    """
    # Define metric categories and values
    categories = ['Returns', 'Returns', 'Returns', 'Risk', 'Risk', 'Trading', 'Trading', 'Trading']
    metric_names = [
        'Total Return', 'CAGR', 'Sharpe Ratio',
        'Max Drawdown', 'Profit Factor',
        'Total Trades', 'Win Rate', 'Avg Days Held'
    ]
    metric_values = [
        f"{metrics.get('total_return_pct', 0):.2f}%",
        f"{metrics.get('cagr', 0):.2f}%",
        f"{metrics.get('sharpe_ratio', 0):.2f}",
        f"{metrics.get('max_drawdown_pct', 0):.2f}%",
        f"{metrics.get('profit_factor', 0):.2f}",
        f"{metrics.get('total_trades', 0)}",
        f"{metrics.get('win_rate', 0):.1f}%",
        f"{metrics.get('avg_days_held', 0):.1f}"
    ]
    
    # Color code based on good/bad values
    def get_color(name, value):
        try:
            v = float(value.replace('%', '').replace('$', '').replace(',', ''))
        except:
            return '#16213e'
        
        if name in ['Total Return', 'CAGR', 'Sharpe Ratio', 'Profit Factor']:
            return '#1b4332' if v > 0 else '#4a1c1c' if v < 0 else '#16213e'
        elif name == 'Max Drawdown':
            return '#1b4332' if v < 10 else '#4a1c1c' if v > 25 else '#16213e'
        elif name == 'Win Rate':
            return '#1b4332' if v > 50 else '#4a1c1c' if v < 40 else '#16213e'
        return '#16213e'
    
    value_colors = [get_color(n, v) for n, v in zip(metric_names, metric_values)]
    
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=['Category', 'Metric', 'Value'],
            fill_color='#1a1a2e',
            font=dict(color='white', size=13),
            align='center',
            height=35
        ),
        cells=dict(
            values=[categories, metric_names, metric_values],
            fill_color=[
                ['#0f3460'] * len(categories),
                ['#16213e'] * len(metric_names),
                value_colors
            ],
            font=dict(color='white', size=12),
            align=['center', 'left', 'center'],
            height=32
        )
    )])
    
    fig.update_layout(
        title=dict(
            text=f"Performance Summary: {market_name}",
            font=dict(size=16, color='#333')
        ),
        height=350,
        margin=dict(l=20, r=20, t=50, b=20)
    )
    
    return fig


# Display performance summary
summary_table = create_metrics_summary(metrics, test_market)
summary_table.show()


In [32]:
def create_equity_curve(equity_curve, trades_df, initial_capital, market_name):
    """
    Create an equity curve chart with drawdown shading.
    
    Args:
        equity_curve: List of equity values over time
        trades_df: DataFrame with trade records (for date range)
        initial_capital: Starting capital
        market_name: Name of market for title
    
    Returns:
        Plotly figure object
    """
    # Create date index based on trade dates or use simple index
    if not trades_df.empty and 'entry_date' in trades_df.columns:
        start_date = trades_df['entry_date'].min()
        # Create date range matching equity curve length
        dates = pd.date_range(start=start_date, periods=len(equity_curve), freq='D')
    else:
        dates = list(range(len(equity_curve)))
    
    equity_series = pd.Series(equity_curve, index=dates)
    
    # Calculate drawdown
    rolling_max = equity_series.cummax()
    drawdown = (equity_series - rolling_max) / rolling_max * 100
    
    # Create figure with secondary y-axis
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    # Equity curve
    fig.add_trace(
        go.Scatter(
            x=dates,
            y=equity_curve,
            name="Equity",
            line=dict(color="#2962FF", width=2),
            fill='tozeroy',
            fillcolor='rgba(41, 98, 255, 0.1)'
        ),
        secondary_y=False
    )
    
    # Initial capital reference line
    fig.add_hline(
        y=initial_capital, 
        line_dash="dash", 
        line_color="gray", 
        line_width=1,
        annotation_text=f"Initial: ${initial_capital:,.0f}",
        annotation_position="right"
    )
    
    # Drawdown on secondary axis
    fig.add_trace(
        go.Scatter(
            x=dates,
            y=drawdown,
            name="Drawdown %",
            line=dict(color="#FF1744", width=1),
            fill='tozeroy',
            fillcolor='rgba(255, 23, 68, 0.2)'
        ),
        secondary_y=True
    )
    
    # Layout
    fig.update_layout(
        title=dict(
            text=f"Equity Curve: {market_name}",
            font=dict(size=16)
        ),
        height=450,
        hovermode="x unified",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        ),
        template="plotly_white"
    )
    
    # Axis labels
    fig.update_yaxes(title_text="Equity ($)", secondary_y=False)
    fig.update_yaxes(title_text="Drawdown (%)", secondary_y=True, range=[-50, 5])
    fig.update_xaxes(title_text="Date")
    
    return fig


# Display equity curve
equity_chart = create_equity_curve(results['equity_curve'], results['trades'], backtester.initial_capital, test_market)
equity_chart.show()


In [33]:
def run_full_backtest(df, market_name, initial_capital=30000, risk_per_trade=0.01,
                      atr_period=10, atr_stop_mult=2, atr_target_mult=3, 
                      max_hold_days=20, rsi_exit=60,
                      commercial_long=80, commercial_short=20, rsi_oversold=30, rsi_overbought=70):
    """
    Run complete backtest pipeline for a single market and display all visualizations.
    
    Args:
        df: Main DataFrame with all market data
        market_name: Name of market to backtest
        initial_capital: Starting capital
        risk_per_trade: Fraction of capital to risk (0.01 = 1%)
        atr_period: ATR calculation period
        atr_stop_mult: ATR multiplier for stop loss
        atr_target_mult: ATR multiplier for take profit
        max_hold_days: Maximum days to hold a trade
        rsi_exit: RSI level to exit trades
        commercial_long: Commercial Index threshold for long bias
        commercial_short: Commercial Index threshold for short bias
        rsi_oversold: RSI oversold threshold
        rsi_overbought: RSI overbought threshold
    
    Returns:
        dict with all results
    """
    print("=" * 80)
    print(f"FULL BACKTEST: {market_name}")
    print("=" * 80)
    
    # Step 1: Prepare data
    print("\n[1/5] Preparing data...")
    data = prepare_strategy_data(df, market_name)
    if data.empty:
        print(f"❌ No data available for {market_name}")
        return None
    
    # Step 2: Calculate ATR
    print("\n[2/5] Calculating ATR...")
    data = calculate_atr(data, period=atr_period)
    
    # Step 3: Generate signals
    print("\n[3/5] Generating signals...")
    data = generate_signals(data, commercial_long, commercial_short, rsi_oversold, rsi_overbought)
    
    # Step 4: Run backtest
    print("\n[4/5] Running backtest...")
    backtester = COTRSIBacktester(
        initial_capital=initial_capital,
        risk_per_trade=risk_per_trade,
        atr_stop_mult=atr_stop_mult,
        atr_target_mult=atr_target_mult,
        max_hold_days=max_hold_days,
        rsi_exit=rsi_exit
    )
    results = backtester.backtest(data, market_name=market_name)
    
    # Step 5: Calculate metrics
    print("\n[5/5] Calculating performance metrics...")
    metrics = calculate_performance_metrics(results['trades'], results['equity_curve'], initial_capital)
    
    # Print summary
    print("\n" + "=" * 80)
    print("RESULTS SUMMARY")
    print("=" * 80)
    print(f"Initial Capital:  ${initial_capital:,.2f}")
    print(f"Final Capital:    ${results['final_capital']:,.2f}")
    print(f"Total Return:     {metrics['total_return_pct']:.2f}%")
    print(f"CAGR:             {metrics['cagr']:.2f}%")
    print(f"Max Drawdown:     {metrics['max_drawdown_pct']:.2f}%")
    print(f"Sharpe Ratio:     {metrics['sharpe_ratio']:.2f}")
    print(f"Total Trades:     {metrics['total_trades']}")
    print(f"Win Rate:         {metrics['win_rate']:.1f}%")
    print(f"Profit Factor:    {metrics['profit_factor']:.2f}")
    
    # Display visualizations
    print("\n📊 Generating visualizations...")
    
    # Strategy chart
    strategy_chart = create_strategy_chart(data, results['trades'], market_name)
    strategy_chart.show()
    
    # Trade table
    if not results['trades'].empty:
        trade_table = create_trade_table(results['trades'], last_n=10)
        trade_table.show()
    
    # Metrics summary
    summary_table = create_metrics_summary(metrics, market_name)
    summary_table.show()
    
    # Equity curve
    equity_chart = create_equity_curve(results['equity_curve'], results['trades'], initial_capital, market_name)
    equity_chart.show()
    
    return {
        'data': data,
        'results': results,
        'metrics': metrics,
        'backtester': backtester
    }


# List available markets
print("Available markets:")
markets = df['Market'].unique()
for i, m in enumerate(markets[:20]):  # Show first 20
    print(f"  {i+1}. {m}")
if len(markets) > 20:
    print(f"  ... and {len(markets) - 20} more")


Available markets:
  1. AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE
  2. BITCOIN - CHICAGO MERCANTILE EXCHANGE
  3. BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE
  4. BRITISH POUND - CHICAGO MERCANTILE EXCHANGE
  5. CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE
  6. COCOA - ICE FUTURES U.S.
  7. COFFEE C - ICE FUTURES U.S.
  8. COPPER- #1 - COMMODITY EXCHANGE INC.
  9. CORN - CHICAGO BOARD OF TRADE
  10. COTTON NO. 2 - ICE FUTURES U.S.
  11. E-MINI NATURAL GAS - NEW YORK MERCANTILE EXCHANGE
  12. E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE
  13. EMINI RUSSELL 1000 GROWTH - CHICAGO MERCANTILE EXCHANGE
  14. ETHER CASH SETTLED - CHICAGO MERCANTILE EXCHANGE
  15. EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE
  16. FEEDER CATTLE - CHICAGO MERCANTILE EXCHANGE
  17. GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE
  18. GOLD - COMMODITY EXCHANGE INC.
  19. JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE
  20. LEAN HOGS - CHICAGO MERCANTILE EXCHANGE
  ... and 27 more


In [34]:
# ============================================================================
# RUN FULL BACKTEST ON A MARKET
# ============================================================================
# Change the market name below to test different markets

selected_market = "GOLD - COMMODITY EXCHANGE INC."  # Change this to test other markets

# Run the full backtest with all visualizations
backtest_results = run_full_backtest(
    df=df,
    market_name=selected_market,
    initial_capital=30000,
    risk_per_trade=0.01,      # 1% risk per trade
    atr_period=10,            # ATR lookback
    atr_stop_mult=2,          # Stop loss = 2x ATR
    atr_target_mult=3,        # Take profit = 3x ATR
    max_hold_days=20,         # Maximum holding period
    rsi_exit=60,              # Exit when RSI crosses 60
    commercial_long=80,       # Long when Commercial Index >= 80
    commercial_short=20,      # Short when Commercial Index <= 20
    rsi_oversold=30,          # RSI oversold level
    rsi_overbought=70         # RSI overbought level
)


FULL BACKTEST: GOLD - COMMODITY EXCHANGE INC.

[1/5] Preparing data...
✓ GOLD - COMMODITY EXCHANGE INC.: 502 rows, columns: ['Date', 'Open', 'High', 'Low', 'Close', 'RSI', 'Net Commercial Position', 'OI', 'Commercial_Index']

[2/5] Calculating ATR...
✓ ATR calculated: 493/502 rows have valid ATR (first 10 rows are NaN)

[3/5] Generating signals...
✓ Signals generated:
   Long signals:  2 (Commercial >= 80 AND RSI < 30)
   Short signals: 50 (Commercial <= 20 AND RSI > 70)
   No signal:     450

[4/5] Running backtest...

[5/5] Calculating performance metrics...

RESULTS SUMMARY
Initial Capital:  $30,000.00
Final Capital:    $29,096.26
Total Return:     -3.01%
CAGR:             -1.90%
Max Drawdown:     6.30%
Sharpe Ratio:     -1.49
Total Trades:     15
Win Rate:         46.7%
Profit Factor:    0.60

📊 Generating visualizations...
